# PubMed LLM Maintenance Runner

Use this notebook for routine lab maintenance. It is designed so you only edit the small settings cell below, then run the task cell you need.

Typical tasks:

- check queue/database status
- process newly requested genes
- retry failed genes
- refresh existing genes monthly
- update code files from GitHub

Do not paste real passwords or service-account JSON into this notebook. Use Colab Secrets or Hugging Face Space secrets.

## 1. Setup

Run this once at the start of a Colab session.

In [ ]:
from google.colab import drive, userdata
import os, subprocess, shlex

drive.mount('/content/drive')
%cd /content/drive/MyDrive/pubmed_llm

# Optional secrets. These are safe to leave empty if not configured.
for name in ['HF_TOKEN', 'GOOGLE_SERVICE_ACCOUNT_JSON', 'GOOGLE_DRIVE_DB_FILE_ID', 'ENTREZ_EMAIL']:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            print(f'Loaded secret: {name}')
    except Exception:
        pass

def run(command):
    print('\n$ ' + command, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}')

run('pip install -q -r requirements-worker.txt')

# Optional Hugging Face login for fewer model-download rate-limit issues.
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(os.environ['HF_TOKEN'])

print('Setup complete.')

## 2. Edit These Settings

Change only these numbers for normal use.

In [ ]:
# Queue processing settings
MAX_REQUESTS = 5      # how many queue/error genes to process in one run
MAX_PAPERS   = 50     # how many new papers to check per gene

# Monthly refresh settings
START_AT  = 0         # stable alphabetical start index for existing genes
MAX_GENES = 5         # number of existing genes to refresh

# Common paths. Usually do not change these.
DB_PATH = '/content/drive/MyDrive/pubmed_llm/gene_function_lab/gene_function_lab.db'
CACHE_DIR = '/content/drive/MyDrive/pubmed_llm/functional_study_cache'

print('Settings loaded:')
print('MAX_REQUESTS =', MAX_REQUESTS)
print('MAX_PAPERS   =', MAX_PAPERS)
print('START_AT     =', START_AT)
print('MAX_GENES    =', MAX_GENES)

## 3. Check Status

Run this before and after every maintenance task.

In [ ]:
run(f'python -u scripts/check_queue_status.py --db-path {shlex.quote(DB_PATH)}')

## 4. Process New Queue Requests

Use this for normal pending genes requested from the website. If a previous run crashed and status shows `processing > 0`, set `RESET_PROCESSING = True` before running.

In [ ]:
RESET_PROCESSING = False

cmd = (
    f'python -u scripts/process_queue.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--cache-dir {shlex.quote(CACHE_DIR)} '
    f'--max-requests {MAX_REQUESTS} '
    f'--max-papers {MAX_PAPERS} '
    f'--upload-at-end'
)
if RESET_PROCESSING:
    cmd += ' --reset-processing'

run(cmd)

## 5. Retry Error Genes

Use this for rows marked `error`, usually from temporary PubMed/network failures. Start small, for example `MAX_REQUESTS = 3` or `5`.

In [ ]:
cmd = (
    f'python -u scripts/process_queue.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--cache-dir {shlex.quote(CACHE_DIR)} '
    f'--retry-errors '
    f'--max-requests {MAX_REQUESTS} '
    f'--max-papers {MAX_PAPERS} '
    f'--upload-at-end'
)
run(cmd)

## 6. Monthly Refresh Existing Genes

Use this once per month to check already-processed genes for newly published papers. Run in stable alphabetical chunks by changing `START_AT`: 0, 15, 30, ...

In [ ]:
cmd = (
    f'python -u scripts/update_existing_genes.py '
    f'--db-path {shlex.quote(DB_PATH)} '
    f'--cache-dir {shlex.quote(CACHE_DIR)} '
    f'--start-at {START_AT} '
    f'--max-genes {MAX_GENES} '
    f'--max-papers {MAX_PAPERS} '
    f'--upload'
)
run(cmd)

## 7. Update Code From GitHub

Run this when GitHub has a new fix. It updates code files only. It does not replace your database.

In [ ]:
BASE = 'https://raw.githubusercontent.com/rf2960/pubmed-llm/main'
files = [
    'pipeline.py',
    'db.py',
    'drive_sync.py',
    'requirements-worker.txt',
    'scripts/common.py',
    'scripts/check_queue_status.py',
    'scripts/process_queue.py',
    'scripts/update_existing_genes.py',
]
run('mkdir -p scripts')
for f in files:
    run(f'curl -L -o {shlex.quote(f)} {BASE}/{f}')
print('Code update complete. Run setup again if requirements changed.')

## 8. Force Drive Upload

Only use this if `GOOGLE_SERVICE_ACCOUNT_JSON` is configured in Colab Secrets. If not, manually replace the shared Drive DB file after processing.

In [ ]:
if not os.environ.get('GOOGLE_SERVICE_ACCOUNT_JSON'):
    print('GOOGLE_SERVICE_ACCOUNT_JSON is not configured. Manual DB replacement is needed.')
else:
    import drive_sync
    drive_sync._drive_file_id = None
    ok = drive_sync.upload_db_to_drive(DB_PATH)
    print('Upload result:', ok)

## 9. Final Checklist

After processing:

1. Run **Check Status** again.
2. If API upload is not configured, replace the shared Drive database file manually.
3. Force website sync with `/api/sync?token=...`.
4. Refresh the website and confirm paper/gene counts changed.
5. Keep a backup copy of the latest database.